# 面试问题：怎样从零实现 GraphSAGE，并对训练时未见的新节点做归纳推理？

## 可直接复述的回答主线

1. GraphSAGE 不为每个节点直接学习独立 embedding，而是学习可复用的 self 与 neighbor 聚合函数。
2. Mean aggregator 对目标节点的邻居特征求均值，再分别经过 W_self 与 W_neigh，合并后得到新表示。
3. 训练后只要新节点有相同 schema 的自身特征和邻居，就能复用参数生成 embedding，这体现结构上的 inductive 能力。
4. 大图中需逐层按 fanout 采邻居，并固定采样 seed、边方向和去重规则，避免训练/服务不一致。
5. 评测应把训练基础图与推理扩展图分开，逐个展示 unseen 节点的原始 baseline、邻居、均值、隐藏表示和预测。
6. 孤立节点的 neighbor mean 必须有显式零向量 fallback，直接除以零会产生 NaN；self 分支仍可提供信息。
7. 生产还需多层 block sampler、特征时点一致性、冷启动策略、动态图缓存、异构边和分布漂移监控。

下面用同一批可读输入依次验证朴素基线、手写核心机制、中间过程、失败修正与生产边界。

## 1. 真实案例与输入预览

案例是商品同购图：训练期有 10 个基础商品，数码和运动各 5 个；推理期新增手机壳、相机包、护腕和运动袜 4 个从未参与训练的商品，并只提供自身特征及与已知商品的同购边。任务是验证同一 GraphSAGE 参数能否分类这些新节点。

In [1]:
import math  # 汇总梯度并计算节点分类指标。
import warnings  # 过滤本地 PyTorch 环境的无关兼容警告。
warnings.filterwarnings("ignore", message=".*pynvml.*", category=FutureWarning)  # 保持输出聚焦 GraphSAGE。
import torch  # 使用基础张量与线性层手写邻居聚合。
torch.manual_seed(261)  # 固定模型、采样和训练轨迹。
torch.set_num_threads(1)  # 固定 CPU 单线程提高可复现性。
class_names = ["数码", "运动"]  # 定义两个商品社区类别。
base_names = ["手机", "耳机", "充电器", "平板", "键盘", "瑜伽垫", "哑铃", "跑鞋", "水杯", "弹力带"]  # 定义训练期十个基础商品。
base_labels = torch.tensor([0] * 5 + [1] * 5, dtype=torch.long)  # 保存基础节点类别。
anchor_signal = [1.2, 0.8, 0.0, 0.0, 0.0, -1.2, -0.8, 0.0, 0.0, 0.0]  # 只给每个社区两个明显锚点特征。
repeated_popularity = [0.2, 0.4, 0.6, 0.3, 0.5] * 2  # 为两类使用相同流行度模式。
base_features = torch.tensor([[anchor_signal[index], repeated_popularity[index], 1.0] for index in range(10)], dtype=torch.float32)  # 构造锚点、流行度和 bias 三维特征。
base_edges = []  # 保存训练期无向同购边。
for offset in [0, 5]:  # 分别构造数码和运动社区。
    for left in range(offset, offset + 5):  # 遍历当前社区左端点。
        for right in range(left + 1, offset + 5):  # 把五个商品组成稠密同购社区。
            base_edges.append((left, right))  # 保存当前社区无向边。
base_edges.append((4, 8))  # 加入键盘与水杯这一条弱跨类同购边。
unseen_names = ["手机壳", "相机包", "护腕", "运动袜"]  # 定义训练时完全不存在的四个新品。
unseen_labels = torch.tensor([0, 0, 1, 1], dtype=torch.long)  # 保存仅用于离线评测的新品真值。
unseen_features = torch.tensor([[0.0, 0.35, 1.0], [0.0, 0.45, 1.0], [0.0, 0.35, 1.0], [0.0, 0.45, 1.0]], dtype=torch.float32)  # 让新品自身特征对两类完全对称。
unseen_edges = [(10, 0), (10, 1), (10, 2), (11, 1), (11, 3), (11, 4), (12, 5), (12, 6), (12, 9), (13, 6), (13, 7), (13, 8)]  # 把每个新品连接到三个同类已知商品。
print("教学实验输入：训练期基础商品图")  # 标记下方为基础图快照。
for index, name in enumerate(base_names):  # 逐基础节点展示特征和邻居。
    neighbors = [base_names[right] if left == index else base_names[left] for left, right in base_edges if left == index or right == index]  # 收集当前基础商品同购邻居。
    print(f"base {index:02d} {name:<5} class={class_names[base_labels[index]]} features={base_features[index].tolist()} neighbors={neighbors}")  # 输出基础节点字段。
print("推理期新增节点")  # 标记下方为训练时未见新品。
for offset, name in enumerate(unseen_names):  # 逐新品展示自身特征和已知邻居。
    node_index = 10 + offset  # 计算扩展图中的新品编号。
    neighbors = [base_names[right] if left == node_index else base_names[left] for left, right in unseen_edges if left == node_index or right == node_index]  # 收集当前新品同购商品。
    print(f"unseen {node_index:02d} {name:<5} true={class_names[unseen_labels[offset]]} features={unseen_features[offset].tolist()} neighbors={neighbors}")  # 输出新品真实推理输入。

教学实验输入：训练期基础商品图
base 00 手机    class=数码 features=[1.2000000476837158, 0.20000000298023224, 1.0] neighbors=['耳机', '充电器', '平板', '键盘']
base 01 耳机    class=数码 features=[0.800000011920929, 0.4000000059604645, 1.0] neighbors=['手机', '充电器', '平板', '键盘']
base 02 充电器   class=数码 features=[0.0, 0.6000000238418579, 1.0] neighbors=['手机', '耳机', '平板', '键盘']
base 03 平板    class=数码 features=[0.0, 0.30000001192092896, 1.0] neighbors=['手机', '耳机', '充电器', '键盘']
base 04 键盘    class=数码 features=[0.0, 0.5, 1.0] neighbors=['手机', '耳机', '充电器', '平板', '水杯']
base 05 瑜伽垫   class=运动 features=[-1.2000000476837158, 0.20000000298023224, 1.0] neighbors=['哑铃', '跑鞋', '水杯', '弹力带']
base 06 哑铃    class=运动 features=[-0.800000011920929, 0.4000000059604645, 1.0] neighbors=['瑜伽垫', '跑鞋', '水杯', '弹力带']
base 07 跑鞋    class=运动 features=[0.0, 0.6000000238418579, 1.0] neighbors=['瑜伽垫', '哑铃', '水杯', '弹力带']
base 08 水杯    class=运动 features=[0.0, 0.30000001192092896, 1.0] neighbors=['瑜伽垫', '哑铃', '跑鞋', '弹力带', '键盘']
base 09 弹力带   class=运动 feature

## 2. Baseline / 基线：只看新品自身特征的最近中心

用十个基础节点计算两类原始特征中心，再对四个新品分类。新品第一维都为零、流行度成对相同，因此自身特征无法区分两类，并列时固定选数码类。

In [2]:
base_centroids = torch.stack([base_features[base_labels == class_index].mean(dim=0) for class_index in range(2)])  # 仅用训练期基础节点计算类别中心。
unseen_baseline_distances = ((unseen_features[:, None, :] - base_centroids[None, :, :]) ** 2).sum(dim=2)  # 计算四个新品到基础类别中心的距离。
unseen_baseline_predictions = unseen_baseline_distances.argmin(dim=1)  # 对对称并列距离固定选择数码类。
unseen_baseline_accuracy = float((unseen_baseline_predictions == unseen_labels).to(torch.float32).mean().item())  # 计算新品自身特征基线准确率。
print("Baseline：unseen raw-feature nearest centroid")  # 标记下表为不读取新品邻居的方案。
print("unseen   true   features            distances                 prediction")  # 输出逐新品基线表头。
for index, name in enumerate(unseen_names):  # 逐新品展示原始特征判断。
    print(f"{name:<8} {class_names[unseen_labels[index]]:<5} {unseen_features[index].tolist()!s:<20} {unseen_baseline_distances[index].tolist()} {class_names[unseen_baseline_predictions[index]]}")  # 输出真值、特征、距离与预测。
print(f"Baseline unseen accuracy={unseen_baseline_accuracy:.4f}")  # 展示冷启动只看自身字段的限制。

Baseline：unseen raw-feature nearest centroid
unseen   true   features            distances                 prediction
手机壳      数码    [0.0, 0.3499999940395355, 1.0] [0.16250000894069672, 0.16250000894069672] 数码
相机包      数码    [0.0, 0.44999998807907104, 1.0] [0.16250000894069672, 0.16250000894069672] 数码
护腕       运动    [0.0, 0.3499999940395355, 1.0] [0.16250000894069672, 0.16250000894069672] 数码
运动袜      运动    [0.0, 0.44999998807907104, 1.0] [0.16250000894069672, 0.16250000894069672] 数码
Baseline unseen accuracy=0.5000


## 3. 底层实现：有向 edge_index、Mean Aggregator、SAGEConv 与确定性采样

无向边显式展开为 source→target 两个方向。`mean_neighbors` 用 `index_add_` 累加指向每个 target 的 source 特征；`SAGEConv` 分别变换 self 与 neighbor mean。采样器按 target 固定 seed 选择至多 fanout 个邻居。

In [3]:
def directed_edge_index(node_count, undirected_edges):  # 把无向边展开为 source-target 两行索引。
    directed_edges = []  # 保存两个方向的边对。
    for left, right in undirected_edges:  # 逐无向边生成双向消息。
        directed_edges.extend([(left, right), (right, left)])  # 同时加入 left→right 与 right→left。
    directed_edges = sorted(set(directed_edges), key=lambda pair: (pair[1], pair[0]))  # 按 target、source 排序保证确定性。
    return torch.tensor(directed_edges, dtype=torch.long).T.contiguous()  # 返回二乘边数的 edge_index。
def mean_neighbors(node_features, edge_index):  # 手写按 target 聚合的邻居均值。
    source, target = edge_index  # 解包 source 与 target 行。
    sums = torch.zeros_like(node_features)  # 初始化每个 target 的特征和。
    counts = torch.zeros(node_features.shape[0], dtype=node_features.dtype)  # 初始化每个 target 邻居计数。
    if source.numel() > 0:  # 检查当前图是否至少有一条边。
        sums.index_add_(0, target, node_features[source])  # 把 source 特征累加到对应 target。
        counts.index_add_(0, target, torch.ones_like(target, dtype=node_features.dtype))  # 累加每个 target 的邻居数。
    means = sums / counts.clamp_min(1.0).unsqueeze(1)  # 对孤立节点用分母一得到显式零均值。
    return means, counts  # 返回邻居均值与计数供解释。
def sample_neighbors(edge_index, node_count, fanout, seed):  # 按 target 手写确定性固定 fanout 采样。
    by_target = [[] for _ in range(node_count)]  # 为每个 target 初始化 source 列表。
    for source, target in edge_index.T.tolist():  # 遍历全部有向消息边。
        by_target[target].append(source)  # 把 source 加入对应 target 邻域。
    sampled = []  # 保存采样后的 source-target 对。
    for target, sources in enumerate(by_target):  # 逐 target 处理邻居集合。
        sources = sorted(set(sources))  # 去重并固定候选顺序。
        if len(sources) > fanout:  # 仅在邻居数超过 fanout 时采样。
            generator = torch.Generator().manual_seed(seed + 1009 * target)  # 为每个 target 派生独立随机流。
            positions = torch.randperm(len(sources), generator=generator)[:fanout].tolist()  # 选择固定数量候选下标。
            sources = sorted(sources[position] for position in positions)  # 恢复稳定 source 顺序。
        sampled.extend((source, target) for source in sources)  # 保存当前 target 的采样消息边。
    return torch.tensor(sampled, dtype=torch.long).T.contiguous() if sampled else torch.empty(2, 0, dtype=torch.long)  # 返回采样 edge_index 或合法空图。
class SAGEConv(torch.nn.Module):  # 定义 self 与 neighbor 双分支 GraphSAGE 层。
    def __init__(self, input_features, output_features):  # 初始化两组独立线性变换。
        super().__init__()  # 注册层参数。
        self.self_linear = torch.nn.Linear(input_features, output_features, bias=False)  # 变换目标节点自身特征。
        self.neighbor_linear = torch.nn.Linear(input_features, output_features, bias=True)  # 变换邻居均值并提供共享 bias。
    def forward(self, node_features, edge_index, return_debug=False):  # 聚合邻居并合并 self 分支。
        neighbor_mean, neighbor_count = mean_neighbors(node_features, edge_index)  # 计算每个 target 的邻居均值与度数。
        self_part = self.self_linear(node_features)  # 计算节点自身贡献。
        neighbor_part = self.neighbor_linear(neighbor_mean)  # 计算聚合邻居贡献。
        output = self_part + neighbor_part  # 相加得到当前层新节点表示。
        debug = {"neighbor_mean": neighbor_mean, "neighbor_count": neighbor_count, "self_part": self_part, "neighbor_part": neighbor_part}  # 汇总消息聚合中间量。
        return (output, debug) if return_debug else output  # 按需返回邻居路径证据。
class GraphSAGE(torch.nn.Module):  # 定义两层可复用聚合器的节点分类网络。
    def __init__(self):  # 初始化三维到八维再到两类的网络。
        super().__init__()  # 注册两层 SAGE 参数。
        self.layer_one = SAGEConv(3, 8)  # 创建第一层邻居特征编码。
        self.layer_two = SAGEConv(8, 2)  # 创建第二层类别 logits 输出。
    def forward(self, node_features, edge_index, return_debug=False):  # 执行两跳 mean GraphSAGE。
        first_output, first_debug = self.layer_one(node_features, edge_index, return_debug=True)  # 计算第一层 self 与 neighbor 分支。
        hidden = torch.relu(first_output)  # 对合并结果执行非线性。
        hidden = torch.nn.functional.normalize(hidden, p=2, dim=1, eps=1.0e-8)  # 逐节点 L2 归一化隐藏表示。
        logits, second_debug = self.layer_two(hidden, edge_index, return_debug=True)  # 再聚合一跳并输出两类分数。
        debug = {"first_neighbor_mean": first_debug["neighbor_mean"], "first_neighbor_count": first_debug["neighbor_count"], "hidden": hidden, "second_neighbor_mean": second_debug["neighbor_mean"]}  # 汇总两层邻域证据。
        return (logits, debug) if return_debug else logits  # 按需返回中间节点表示。
base_edge_index = directed_edge_index(len(base_names), base_edges)  # 构造训练期十节点双向 edge_index。
sampled_base_edges = sample_neighbors(base_edge_index, len(base_names), fanout=3, seed=261)  # 为训练演示生成固定三邻居采样图。
model = GraphSAGE()  # 创建待训练两层 GraphSAGE。
optimizer = torch.optim.Adam(model.parameters(), lr=0.035)  # 创建基础图节点分类优化器。
history = []  # 保存真实 backward 的损失与梯度轨迹。
for step in range(240):  # 只在十个基础节点和训练期图上优化参数。
    optimizer.zero_grad(set_to_none=True)  # 清除上一步全部参数梯度。
    logits, training_debug = model(base_features, base_edge_index, return_debug=True)  # 对基础图执行两层 mean 聚合。
    loss = torch.nn.functional.cross_entropy(logits, base_labels)  # 计算十个基础节点监督交叉熵。
    loss.backward()  # 对 self 与 neighbor 两组权重真实反向传播。
    gradient_norm = math.sqrt(sum(float((parameter.grad ** 2).sum().item()) for parameter in model.parameters() if parameter.grad is not None))  # 汇总全部非空梯度二范数。
    optimizer.step()  # 应用 Adam 更新可复用聚合参数。
    if step % 60 == 0 or step == 239:  # 每六十步保存训练证据。
        accuracy = float((logits.argmax(dim=1) == base_labels).to(torch.float32).mean().item())  # 计算基础图节点准确率。
        history.append({"step": step, "loss": loss.item(), "base_accuracy": accuracy, "gradient_norm": gradient_norm})  # 保存损失、准确率和梯度。
expanded_names = base_names + unseen_names  # 构造推理期十四节点名称列表。
expanded_features = torch.cat([base_features, unseen_features], dim=0)  # 拼接训练期和新品特征。
expanded_labels = torch.cat([base_labels, unseen_labels], dim=0)  # 拼接仅供离线评测的真值。
expanded_edges = base_edges + unseen_edges  # 构造包含新品同购关系的推理图。
expanded_edge_index = directed_edge_index(len(expanded_names), expanded_edges)  # 构造十四节点双向消息边。
sampled_expanded_edges = sample_neighbors(expanded_edge_index, len(expanded_names), fanout=2, seed=261)  # 为推理图生成确定性二邻居采样路径。
model.eval()  # 切换到确定性归纳推理模式。
with torch.no_grad():  # 在从未训练过的扩展图上复用相同参数。
    expanded_logits, expanded_debug = model(expanded_features, expanded_edge_index, return_debug=True)  # 为基础与新品同时生成表示和 logits。
print("GraphSAGE训练轨迹=", history)  # 展示只在十基础节点上的 loss 和梯度。
print("base full/sample edges=", base_edge_index.shape[1], sampled_base_edges.shape[1])  # 展示固定 fanout 减少的消息数。
for node_index in range(10, 14):  # 逐新品展示采样后的 source 路径。
    sampled_sources = sampled_expanded_edges[0, sampled_expanded_edges[1] == node_index].tolist()  # 读取指向当前新品的采样邻居。
    print(f"{expanded_names[node_index]} sampled_sources={[expanded_names[source] for source in sampled_sources]}")  # 输出可追溯采样路径。
print("unseen neighbor means=", torch.round(expanded_debug["first_neighbor_mean"][10:14] * 1000) / 1000)  # 展示新品第一层实际聚合特征。
print("unseen hidden=", torch.round(expanded_debug["hidden"][10:14] * 1000) / 1000)  # 展示可复用参数生成的新品 embedding。

GraphSAGE训练轨迹= [{'step': 0, 'loss': 0.7867633104324341, 'base_accuracy': 0.10000000149011612, 'gradient_norm': 0.7715310170162168}, {'step': 60, 'loss': 0.01085578091442585, 'base_accuracy': 1.0, 'gradient_norm': 0.014115587464551943}, {'step': 120, 'loss': 0.005011123139411211, 'base_accuracy': 1.0, 'gradient_norm': 0.00649425486447681}, {'step': 180, 'loss': 0.003069725353270769, 'base_accuracy': 1.0, 'gradient_norm': 0.003962604040227103}, {'step': 239, 'loss': 0.002126406878232956, 'base_accuracy': 1.0, 'gradient_norm': 0.002736253866193694}]
base full/sample edges= 42 30
手机壳 sampled_sources=['手机', '充电器']
相机包 sampled_sources=['耳机', '平板']
护腕 sampled_sources=['瑜伽垫', '弹力带']
运动袜 sampled_sources=['跑鞋', '水杯']
unseen neighbor means= tensor([[ 0.6670,  0.4000,  1.0000],
        [ 0.2670,  0.4000,  1.0000],
        [-0.6670,  0.3670,  1.0000],
        [-0.2670,  0.4330,  1.0000]])
unseen hidden= tensor([[0., 1., 0., 0., 0., 0., 0., 0.],
        [0., 1., 0., 0., 0., 0., 0., 0.],
        [1.,

## 4. 新节点逐样本结果与结果解读

模型训练时只存在前十个节点。现在不更新参数，把四个新品连入扩展图，比较自身特征基线与 GraphSAGE 归纳预测，并输出邻居均值和置信度。

In [4]:
expanded_probabilities = torch.softmax(expanded_logits, dim=1)  # 把十四节点 logits 转为类别概率。
expanded_predictions = expanded_probabilities.argmax(dim=1)  # 取得基础节点和新品最高概率类别。
unseen_sage_predictions = expanded_predictions[10:14]  # 截取四个训练时未见节点预测。
unseen_sage_accuracy = float((unseen_sage_predictions == unseen_labels).to(torch.float32).mean().item())  # 计算新品归纳准确率。
base_in_expanded_accuracy = float((expanded_predictions[:10] == base_labels).to(torch.float32).mean().item())  # 检查扩展图中基础节点结果。
print("unseen   true   raw_baseline  GraphSAGE  confidence  neighbors                  neighbor_mean")  # 输出逐新品归纳表头。
for offset, name in enumerate(unseen_names):  # 逐新品展示 baseline 与邻域模型判断。
    node_index = 10 + offset  # 计算当前新品扩展图编号。
    neighbors = [expanded_names[right] if left == node_index else expanded_names[left] for left, right in expanded_edges if left == node_index or right == node_index]  # 读取当前新品全部同购邻居。
    confidence = float(expanded_probabilities[node_index, expanded_predictions[node_index]].item())  # 读取当前最高类别概率。
    print(f"{name:<8} {class_names[unseen_labels[offset]]:<5} {class_names[unseen_baseline_predictions[offset]]:<12} {class_names[unseen_sage_predictions[offset]]:<9} {confidence:>10.4f} {str(neighbors):<26} {expanded_debug['first_neighbor_mean'][node_index].tolist()}")  # 输出真值、两种预测、邻居和均值。
print(f"结果解读：新品 raw-feature baseline accuracy={unseen_baseline_accuracy:.4f}，不更新参数的GraphSAGE={unseen_sage_accuracy:.4f}；收益来自可复用邻居聚合。")  # 解释结构归纳能力与同数据边界。

unseen   true   raw_baseline  GraphSAGE  confidence  neighbors                  neighbor_mean
手机壳      数码    数码           数码            0.9986 ['手机', '耳机', '充电器']        [0.6666666865348816, 0.4000000059604645, 1.0]
相机包      数码    数码           数码            0.9986 ['耳机', '平板', '键盘']         [0.2666666805744171, 0.4000000059604645, 1.0]
护腕       运动    数码           运动            0.9987 ['瑜伽垫', '哑铃', '弹力带']       [-0.6666666865348816, 0.36666667461395264, 1.0]
运动袜      运动    数码           运动            0.9987 ['哑铃', '跑鞋', '水杯']         [-0.2666666805744171, 0.43333330750465393, 1.0]
结果解读：新品 raw-feature baseline accuracy=0.5000，不更新参数的GraphSAGE=1.0000；收益来自可复用邻居聚合。


## 5. 失败案例与修正：孤立新品邻居数为零

再加入一个没有任何边的未知配件。朴素 `sum/count` 会除以零得到 NaN；安全 aggregator 用 `clamp_min(1)` 令 neighbor mean 为零，GraphSAGE 仍可沿 self 分支输出有限结果。

In [5]:
isolated_feature = torch.tensor([[0.2, 0.4, 1.0]])  # 定义一个无同购边新品的自身特征。
features_with_isolated = torch.cat([expanded_features, isolated_feature], dim=0)  # 把孤立新品加入十五节点特征表。
isolated_edge_index = expanded_edge_index  # 保持原边集合使第十五个节点完全无邻居。
safe_means, safe_counts = mean_neighbors(features_with_isolated, isolated_edge_index)  # 用安全 aggregator 计算包含孤立点的均值。
source, target = isolated_edge_index  # 解包消息边供错误实现复现。
naive_sums = torch.zeros_like(features_with_isolated)  # 初始化错误实现的邻居特征和。
naive_counts = torch.zeros(features_with_isolated.shape[0])  # 初始化错误实现的邻居计数。
naive_sums.index_add_(0, target, features_with_isolated[source])  # 累加已有边的 source 特征。
naive_counts.index_add_(0, target, torch.ones_like(target, dtype=torch.float32))  # 累加已有 target 的度数。
naive_means = naive_sums / naive_counts.unsqueeze(1)  # 故意直接除零复现孤立点 NaN。
with torch.no_grad():  # 在无梯度环境验证安全网络输出。
    isolated_logits, isolated_debug = model(features_with_isolated, isolated_edge_index, return_debug=True)  # 让同一 GraphSAGE 处理孤立新品。
print(f"错误行为：isolated count={naive_counts[-1].item():.0f}，naive_mean={naive_means[-1].tolist()}，finite={torch.isfinite(naive_means[-1]).all().item()}")  # 展示直接除零产生 NaN。
print(f"修正行为：safe_mean={safe_means[-1].tolist()}，self-only logits={isolated_logits[-1].tolist()}，finite={torch.isfinite(isolated_logits[-1]).all().item()}")  # 展示零邻居 fallback 与 self 分支结果。

错误行为：isolated count=0，naive_mean=[nan, nan, nan]，finite=False
修正行为：safe_mean=[0.0, 0.0, 0.0]，self-only logits=[1.6424106359481812, -1.628690481185913]，finite=True


## 6. 生产边界

十个基础节点和四个新品只验证参数复用。生产需按时间严格隔离训练与新节点、处理特征时点一致性、使用多层 computation block sampler、按边权/类型/时间采样、缓存高频 embedding、定义孤立点与 OOV fallback、校准冷启动置信度，并监控 fanout、邻居缺失率、新品召回率和图快照漂移。

In [6]:
graphsage_diagnostics = {"training_nodes": len(base_names), "unseen_nodes": len(unseen_names), "training_edges": len(base_edges), "expanded_edges": len(expanded_edges), "baseline_unseen_accuracy": unseen_baseline_accuracy, "graphsage_unseen_accuracy": unseen_sage_accuracy, "base_accuracy_after_expansion": base_in_expanded_accuracy, "initial_loss": history[0]["loss"], "final_loss": history[-1]["loss"], "isolated_neighbor_count": float(safe_counts[-1].item()), "isolated_logits_finite": bool(torch.isfinite(isolated_logits[-1]).all())}  # 汇总图快照、训练、归纳与孤立点指标。
print("生产监控快照：", graphsage_diagnostics)  # 输出 GraphSAGE 归纳服务应持续观察的信号。

生产监控快照： {'training_nodes': 10, 'unseen_nodes': 4, 'training_edges': 21, 'expanded_edges': 33, 'baseline_unseen_accuracy': 0.5, 'graphsage_unseen_accuracy': 1.0, 'base_accuracy_after_expansion': 1.0, 'initial_loss': 0.7867633104324341, 'final_loss': 0.002126406878232956, 'isolated_neighbor_count': 0.0, 'isolated_logits_finite': True}


## 7. 最小回归测试

最后一格只保护基础/新品规模、真实训练、确定性采样、归纳收益和孤立点修正。

In [7]:
assert len(base_names) >= 6 and len(unseen_names) >= 4 and base_edge_index.shape[0] == 2  # 保证基础图和新品案例规模充足。
assert history[-1]["loss"] < history[0]["loss"] and all(row["gradient_norm"] > 0.0 for row in history)  # 保证 self/neighbor 参数真实 backward 学习。
assert torch.equal(sampled_expanded_edges, sample_neighbors(expanded_edge_index, len(expanded_names), fanout=2, seed=261))  # 保证固定 fanout 采样逐位可复现。
assert unseen_sage_accuracy > unseen_baseline_accuracy and unseen_sage_accuracy >= 0.75  # 保证训练时未见节点从邻居聚合获得收益。
assert not torch.isfinite(naive_means[-1]).all() and torch.equal(safe_means[-1], torch.zeros(3))  # 保证孤立点除零失败可复现并获得零均值 fallback。
assert safe_counts[-1].item() == 0.0 and torch.isfinite(isolated_logits[-1]).all()  # 保证孤立点通过 self 分支得到有限输出。